In [ ]:
# %load smacg.py
#!/bin/env python
%reload_ext autoreload
%autoreload 2
import numpy as np
from os.path import join
%pylab inline

In [ ]:
filename= '/rfs/data/MERIS/4threprocessing/ftp.acri.fr/MER4VAL/MER4RP_Italy_Land-aero/MEGS_34_37/vicarious_newformat/ENV_ME_2_RRG____20090507T093708_20090507T094045_________________0216_078_437______REM_R_NT____.SEN3/'

from snappy import Product
from snappy import ProductData
from snappy import ProductIO
from snappy import ProductUtils
from snappy import FlagCoding


print("Reading...")
product = ProductIO.readProduct(filename)
width = product.getSceneRasterWidth()
height = product.getSceneRasterHeight()
name = product.getName()
description = product.getDescription()
band_names = product.getBandNames()

print("Product:     %s, %s" % (name, description))
print("Raster size: %d x %d pixels" % (width, height))
print("Start time:  " + str(product.getStartTime()))
print("End time:    " + str(product.getEndTime()))
#print("Bands:       %s" % (list(band_names)))

In [ ]:
NB    = 1
### band settings
# example : all first 10 bands of MERIS
dirname = '/home/did/RTC/smacg'
sensor  = 'MERIS'
aer     = 'DES' # Desert dust aerosol coefficient
bands_coef = [join(dirname,'COEFFS/coef_'+sensor+str(x+8)+'_'+aer+'.dat') for x in range(NB)]

XSIZE = 512
YSIZE = 512
XOFF  = 0
YOFF  = 0
bands = ['M'+'{00:02d}'.format(x+8)+'_rho_TOA' for x in range(NB)]
buff  = np.zeros((YSIZE, XSIZE), dtype='float32') + np.NaN
rtoa  = np.zeros((YSIZE, XSIZE, NB), dtype='float32') 
tetas = np.zeros((YSIZE, XSIZE), dtype='float32')
tetav = np.zeros((YSIZE, XSIZE), dtype='float32')
phis  = np.zeros((YSIZE, XSIZE), dtype='float32')
phiv  = np.zeros((YSIZE, XSIZE), dtype='float32')
uh2o  = np.ones(( YSIZE, XSIZE), dtype='float32') *2.0 + 0.5
uo3   = np.zeros((YSIZE, XSIZE), dtype='float32')*0.1 + 0.2
taup550  = np.zeros((YSIZE, XSIZE), dtype='float32') + 0.1
pression = np.zeros((YSIZE, XSIZE), dtype='float32')
alt   = np.zeros((YSIZE, XSIZE), dtype='float32')

for iband,band in enumerate(bands):
    rad = product.getBand(band)
    rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, buff)
    rtoa[:,:,iband] = buff[:,:]

rad = product.getBand('theta_s')
rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, tetas)
rad = product.getBand('theta_v')
rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, tetav)
rad = product.getBand('delta_phi')
rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, phiv)
rad = product.getBand('p_ecmwf')
rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, pression)
rad = product.getBand('altitude')
rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, alt)
pression *= np.exp(-alt/8000.)
imshow(pression)
figure()
imshow(rtoa[:,:,0],vmax=0.5)

In [ ]:
from smacg import Smacg
S=Smacg()

### GPU grid
XBLOCK  = 128
XGRID   = 256
YGRID   = 1
YBLOCK  = 1

### Data segmentation
M       = XBLOCK * XGRID     # elemnentary size of pixel's array : match the GPU grid
Z       = XSIZE  * YSIZE / M # additionalvi sm 3rd dimension data : each pixel process 
NLOOP   = 1

# Inputs arrays reshaping
rtoa    = rtoa.reshape((XBLOCK,XGRID,Z,NB),order='C')

# Run
res=S.run(bands_coef, tetas, tetav, phis, phiv,
            uh2o, uo3, taup550, pression, rtoa,
            XBLOCK=XBLOCK, XGRID=XGRID, NBLOOP=1)

In [ ]:
rsurf=res[5].reshape((XSIZE, YSIZE, NB),order='C')
rsurf.shape
imshow(rsurf[:,:,0],vmin= -0.5, vmax=0.5)


In [ ]:
f=figure()
imshow((duh2o*0.01).reshape(4096,4096))
figure()
imshow((drtoa*0.01).reshape(4096,4096))
figure()
imshow((dpre*0.01).reshape(4096,4096))
figure()
imshow((dtaup*0.01).reshape(4096,4096))

In [ ]:
from sympy import *
r,ra,t,tts,ttv,r,s,tg = symbols('r ra t tts ttv r s tg')
diff((r-(ra * t))/ ( t * tts * ttv + (r - (ra * t)) * s),ra)    

In [ ]:
from sympy import *
r,ra,t,tts,ttv,r,s, to3 = symbols('r ra t tts ttv r s to3')
diff((r-(ra * tg*to3))/ ( tg*to3 * tts * ttv + (r - (ra * tg*to3)) * s),to3)    

In [ ]:
diff((r-(ra * t))/ ( t * tts * ttv + (r - (ra * t)) * s),r) 

In [ ]:
from sympy import *
r,ra,t,tts,ttv,r,s, ca_ao3, uo3, ca_no3, m = symbols('r ra t tts ttv r s ca_ao3 uo3 ca_no3 m')
diff (exp ( (ca_ao3)  * pow ( (uo3 *m)  , (ca_no3)  ) ), uo3)

In [ ]:
from sympy import *
rtoa, atm_ref, tg, ttetas, ttetav, s, ttt = symbols('rtoa atm_ref tg ttetas ttetav s ttt')
ttt = tg*ttetas*ttetav
rsurf = (rtoa- atm_ref*tg) / (ttt + (rtoa-atm_ref*tg)*s )
diff (rsurf, tg)

In [ ]:
diff (rsurf, rtoa)

In [ ]:
from sympy import *
ao3,m,no3,uo3,to3 = symbols('ao3 m no3 uo3 to3')
to3 = exp(ao3 * (m*uo3)**no3)

In [ ]:
diff(to3, uo3)

In [ ]:
from sympy import *
rtoa, atm_ref, tgp, to3, ttetas, ttetav, s, ttt = symbols('rtoa atm_ref tgp to3 ttetas ttetav s ttt')
ttt = tgp*to3*ttetas*ttetav
rsurf = (rtoa- atm_ref*tgp*to3) / (ttt + (rtoa-atm_ref*tgp*to3)*s )
diff (rsurf, to3)